## Задание 11: NDCG@10

Выведем номера предложений в корпусе и оценим релевантность предложений по шкале {2, 1, 0}:

In [2]:
import re
import math
import numpy as np
from collections import Counter, defaultdict
import pymorphy3
import sentencepiece as spm
import os

In [128]:
from razdel import sentenize

with open("corpus.txt", "r", encoding="utf-8") as f:
    corpus_text = f.read().strip()

corpus_texts = [s.text for s in sentenize(corpus_text)]

In [129]:
CORPUS_SIZE = len(corpus_texts)

In [130]:
for i, sent in enumerate(corpus_texts):
    print(i, "--------------------")
    print(sent)

0 --------------------
Памятник Пушкину на Бауманской улице в Москве — скульптурное изображение (бюст), установленное на месте предполагаемого рождения русского поэта и писателя Александра Сергеевича Пушкина.
1 --------------------
== История ==
Памятник был сооружён вблизи того места, где ранее находился дом по Немецкой (ныне — Бауманская) улице, в котором, как предполагалось, 26 мая (6 июня) 1799 года родился русский поэт и писатель Александр Сергеевич Пушкин (1799—1837).
2 --------------------
Более поздние исследования, проведённые уже после установки памятника, показывают, что место рождения Пушкина скорее находится на соседней Малой Почтовой улице.
3 --------------------
Деревянный дом № 40 был снесён в советское время, на его месте была построена школа № 345, названная именем Пушкина.
4 --------------------
Памятник был открыт 31 октября 1967 года.
5 --------------------
== Описание ==
Памятник был сооружён по проекту скульптора Екатерины Белашовой и архитектора Ю. О. Соколова.


Заведем матрицу релевантностей rel_q и оценим каждое предложения для всех запросов

In [131]:
rel_q = 3 * [[0] * CORPUS_SIZE]

In [132]:
rel_q[0][0] = 2
rel_q[0][1] = 1
rel_q[0][2] = 2
rel_q[0][3] = 1

In [133]:
rel_q[1][73] = 1
rel_q[1][74] = 1
rel_q[1][75] = 2

In [134]:
rel_q[2][185] = 2
rel_q[2][186] = 2
rel_q[2][187] = 1
rel_q[2][188] = 1
rel_q[2][189] = 1
rel_q[2][191] = 1
rel_q[2][192] = 1
rel_q[2][196] = 1
rel_q[2][198] = 1
rel_q[2][200] = 1
rel_q[2][201] = 1
rel_q[2][204] = 1
rel_q[2][206] = 1
rel_q[2][211] = 1
rel_q[2][213] = 1
rel_q[2][225] = 1
rel_q[2][226] = 1
rel_q[2][229] = 1
rel_q[2][236] = 1
rel_q[2][237] = 1
rel_q[2][221] = 1
rel_q[2][230] = 1
rel_q[2][231] = 1
rel_q[2][232] = 2
rel_q[2][233] = 2
rel_q[2][300] = 1

In [135]:
import numpy as np
rel_q_sorted = np.sort(rel_q, axis=1)[:, ::-1]

In [136]:
rel_q_sorted

array([[2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

## Пословная токенизация, unigram, BPE

In [137]:
morph = pymorphy3.MorphAnalyzer()

# пословная токенизация с использованием регулярного выражения
def tokenize_custom(text):
    words = re.findall(r'\b[а-яёa-z\-]+\b', text.lower())
    return [morph.parse(w)[0].normal_form for w in words]


def train_sentencepiece(input_file, model_prefix, model_type="bpe", vocab_size=2000):
    spm.SentencePieceTrainer.train(
        input=input_file,
        model_prefix=model_prefix,
        vocab_size=vocab_size,
        model_type=model_type,
        character_coverage=1.0,
        split_digits=True,
        byte_fallback=False,
    )
    return spm.SentencePieceProcessor(model_file=model_prefix + ".model")

# токенизация при помощи Unigram и BPE
def tokenize_sp(text, sp):
    return sp.encode(text, out_type=str)

def build_vocabulary(tokenized_docs):
    vocab = []
    for doc in tokenized_docs:
        for token in doc:
            if token not in vocab:
                vocab.append(token)

    vocab.sort()
    token2id = {token: i for i, token in enumerate(vocab)}
    return vocab, token2id

def compute_tf_matrix(tokenized_docs, token2id):
    tf = np.zeros((len(tokenized_docs), len(token2id)), dtype=float)
    for i, doc in enumerate(tokenized_docs):
        for term in doc:
            if term in token2id:
                tf[i, token2id[term]] += 1
    return tf

def compute_df(tokenized_docs):
    df = defaultdict(int)
    for doc in tokenized_docs:
        for term in set(doc):
            df[term] += 1
    return df

def compute_tfidf_matrix(tf_matrix, tokenized_docs, token2id):
    N = len(tokenized_docs)
    df = compute_df(tokenized_docs)
    idf = np.zeros(len(token2id), dtype=float)
    for term, idx in token2id.items():
        idf[idx] = math.log(N / (df[term]), 10)
    return tf_matrix * idf

def cosine_similarity(a, b):
    an, bn = np.linalg.norm(a), np.linalg.norm(b)
    if an == 0 or bn == 0:
        return 0.0
    return float(np.dot(a, b) / (an * bn))

# токенизация и векторизация запроса
def vectorize_query(query, token2id, tokenizer_name, sp=None):
    vec = np.zeros(len(token2id), dtype=float)
    if tokenizer_name == "custom":
        tokens = tokenize_custom(query)
    elif tokenizer_name in {"unigram", "bpe"}:
        tokens = sp.encode(query, out_type=str)

    for t in tokens:
        if t in token2id:
            vec[token2id[t]] += 1
    return vec, tokens

# функция поиска
def find_most_relevant(query, q_num, corpus_texts, tokenized_docs, token2id, tf_matrix, tfidf_matrix, tokenizer_name, sp=None, topk=10):
    q_tf, q_tokens = vectorize_query(query, token2id, tokenizer_name, sp)
    df = compute_df(tokenized_docs)
    N = len(tokenized_docs)
    q_tfidf = np.zeros(len(token2id), dtype=float)
    for term, idx in token2id.items():
        if q_tf[idx] > 0:
            idf = math.log(N / (df.get(term, 1)))
            q_tfidf[idx] = q_tf[idx] * idf

    sims_tf = [cosine_similarity(q_tf, tf_matrix[i]) for i in range(N)]
    sims_tfidf = [cosine_similarity(q_tfidf, tfidf_matrix[i]) for i in range(N)]

    print(f"\n===== {tokenizer_name.upper()} =====")
    print("\n--- TF ---")
    count = 0
    rels_tf = [0] * topk
    for i, s in sorted(enumerate(sims_tf), key=lambda x: x[1], reverse=True)[:topk]:
        count += 1
        print(f"{count}. ({s:.4f}) {corpus_texts[i]}")
        rels_tf[count-1] = rel_q[q_num][i]
        
    print("\n--- TF-IDF ---")
    count = 0
    rels_tf_idf = [0] * topk
    for i, s in sorted(enumerate(sims_tfidf), key=lambda x: x[1], reverse=True)[:topk]:
        count += 1
        print(f"{count}. ({s:.4f}) {corpus_texts[i]}")
        rels_tf_idf[count-1] = rel_q[q_num][i]
    return rels_tf, rels_tf_idf

In [138]:
# пословная токенизация
tokenized_custom = [tokenize_custom(t) for t in corpus_texts]
vocab_custom, token2id_custom = build_vocabulary(tokenized_custom)
tf_custom = compute_tf_matrix(tokenized_custom, token2id_custom)
tfidf_custom = compute_tfidf_matrix(tf_custom, tokenized_custom, token2id_custom)

# Unigram
sp_unigram = spm.SentencePieceProcessor(model_file='sp_unigram.model')
tokenized_unigram = [tokenize_sp(t, sp_unigram) for t in corpus_texts]
vocab_unigram, token2id_unigram = build_vocabulary(tokenized_unigram)
tf_unigram = compute_tf_matrix(tokenized_unigram, token2id_unigram)
tfidf_unigram = compute_tfidf_matrix(tf_unigram, tokenized_unigram, token2id_unigram)

# BPE
sp_bpe = spm.SentencePieceProcessor(model_file='sp_bpe.model')
tokenized_bpe = [tokenize_sp(t, sp_bpe) for t in corpus_texts]
vocab_bpe, token2id_bpe = build_vocabulary(tokenized_bpe)
tf_bpe = compute_tf_matrix(tokenized_bpe, token2id_bpe)
tfidf_bpe = compute_tfidf_matrix(tf_bpe, tokenized_bpe, token2id_bpe)

In [139]:
test_queries = [
    "Памятник на месте рождения Пушкина стоит не там.",
    "Одна из самых известных сцен в истории Голливуда происходит на кукурузном поле.",
    "Победный гол сальвадорского футболиста (на илл.) в ворота соперника привёл к шестидневной войне."
]

In [140]:
q_custom_tf, q_custom_tf_idf, q_unigram_tf, q_unigram_tf_idf, q_bpe_tf, q_bpe_tf_idf = [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0]
for i, query in enumerate(test_queries):
    print("\n\nЗапрос: ", query)
    q_custom_tf[i], q_custom_tf_idf[i] = find_most_relevant(query, i, corpus_texts, tokenized_custom, token2id_custom, tf_custom, tfidf_custom, "custom")
    q_unigram_tf[i], q_unigram_tf_idf[i] = find_most_relevant(query, i, corpus_texts, tokenized_unigram, token2id_unigram, tf_unigram, tfidf_unigram, "unigram", sp_unigram)
    q_bpe_tf[i], q_bpe_tf_idf[i] = find_most_relevant(query, i, corpus_texts, tokenized_bpe, token2id_bpe, tf_bpe, tfidf_bpe, "bpe", sp_bpe)
    print("\n---------------------------------------------------------------------------------------------------------")



Запрос:  Памятник на месте рождения Пушкина стоит не там.

===== CUSTOM =====

--- TF ---
1. (0.5189) Памятник Пушкину на Бауманской улице в Москве — скульптурное изображение (бюст), установленное на месте предполагаемого рождения русского поэта и писателя Александра Сергеевича Пушкина.
2. (0.4226) Более поздние исследования, проведённые уже после установки памятника, показывают, что место рождения Пушкина скорее находится на соседней Малой Почтовой улице.
3. (0.2673) Деревянный дом № 40 был снесён в советское время, на его месте была построена школа № 345, названная именем Пушкина.
4. (0.2474) Отличие памятника от многочисленных других памятников Пушкину состоит в том, что он изображает поэта в юном возрасте.
5. (0.2315) В то время Ямайка забила 4 гола (1 место), Кюрасао — 2 мяча (2 место), а Сальвадор (3 место) поразил ворота соперников лишь единожды.
6. (0.2097) В декабре 1969 года на границе между Сальвадором и Гондурасом имели место вооружённые столкновения.
7. (0.2037) == Истор

In [141]:
q_custom_tf

[[2, 2, 1, 0, 0, 0, 1, 0, 1, 0],
 [0, 2, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 1, 1, 0, 0]]

## Fasttext

In [146]:
import numpy as np
import fasttext
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [75]:
corpus = corpus_texts

In [76]:
queries = test_queries

Используем модель fasttext с предобученными векторами "wiki.ru.vec". Файл был загружен по ссылке https://fasttext.cc/docs/en/pretrained-vectors.html

In [72]:
from gensim.models import KeyedVectors

fasttext_path = "/home/aigul/Downloads/wiki.ru/wiki.ru.vec"
ft_model = KeyedVectors.load_word2vec_format(fasttext_path, binary=False)

Создадим TF-IDF векторизатор, обучим его на корпусе и запросах и сформируем словарь слов с соответствующими значениями IDF:

In [77]:
# TF-IDF
vectorizer = TfidfVectorizer()
vectorizer.fit(corpus + queries)
idf_dict = dict(zip(vectorizer.get_feature_names_out(), vectorizer.idf_))

Сформируем матрицы векторов для всего корпуса с использованием и без использования IDF:

In [78]:
def sentence_vector(sentence, use_idf=False):
    words = [w.lower() for w in sentence.split() if w.lower() in ft_model.key_to_index]
    if not words:
        return np.zeros(ft_model.vector_size)

    vectors = []
    for w in words:
        weight = idf_dict.get(w, 1.0) if use_idf else 1.0
        vectors.append(ft_model[w] * weight)
    return np.mean(vectors, axis=0)

corpus_vecs = np.array([sentence_vector(sent, use_idf=False) for sent in corpus])
corpus_vecs_idf = np.array([sentence_vector(sent, use_idf=True) for sent in corpus])

In [79]:
from razdel import sentenize
import numpy as np

corpus_sentences = corpus

def sentence_vector(sentence, use_idf=False):
    words = [w.lower() for w in sentence.split() if w.lower() in ft_model.key_to_index]
    if not words:
        return np.zeros(ft_model.vector_size)

    vectors = []
    for w in words:
        weight = idf_dict.get(w, 1.0) if use_idf else 1.0
        vectors.append(ft_model[w] * weight)
    return np.mean(vectors, axis=0)

corpus_vecs = np.array([sentence_vector(sent, use_idf=False) for sent in corpus_sentences])
corpus_vecs_idf = np.array([sentence_vector(sent, use_idf=True) for sent in corpus_sentences])


In [147]:
topk = 10
q_fasttext_tf, q_fasttext_tf_idf = [0, 0, 0], [0, 0, 0]
# поиск релевантных предложений
for q_num, q in enumerate(queries):
    q_vec = sentence_vector(q, use_idf=False).reshape(1, -1)
    q_vec_idf = sentence_vector(q, use_idf=True).reshape(1, -1)

    sim = cosine_similarity(q_vec, corpus_vecs)[0]
    sim_idf = cosine_similarity(q_vec_idf, corpus_vecs_idf)[0]

    top_idx = np.argsort(sim)[::-1][:top_k]
    top_idx_idf = np.argsort(sim_idf)[::-1][:top_k]

    print("=================================================================")
    print(f"Запрос: {q}")
    print("\nУсреднённые векторы fasttext:")
    rels_tf = [0] * topk
    count = 0
    for i in top_idx:
        print(f"  ({sim[i]:.3f}) {corpus_sentences[i]}")
        rels_tf[count] = rel_q[q_num][i]
        count += 1
    q_fasttext_tf[q_num] = rels_tf   
    print("\nУсреднение с учётом IDF:")
    count = 0
    rels_tf_idf = [0] * topk
    for i in top_idx_idf:
        print(f"  ({sim_idf[i]:.3f}) {corpus_sentences[i]}")
        rels_tf_idf[count] = rel_q[q_num][i]
        count += 1
    q_fasttext_tf_idf[q_num] = rels_tf_idf  

Запрос: Памятник на месте рождения Пушкина стоит не там.

Усреднённые векторы fasttext:
  (0.842) Памятник Пушкину на Бауманской улице в Москве — скульптурное изображение (бюст), установленное на месте предполагаемого рождения русского поэта и писателя Александра Сергеевича Пушкина.
  (0.790) Более поздние исследования, проведённые уже после установки памятника, показывают, что место рождения Пушкина скорее находится на соседней Малой Почтовой улице.
  (0.741) Отличие памятника от многочисленных других памятников Пушкину состоит в том, что он изображает поэта в юном возрасте.
  (0.738) == История ==
Памятник был сооружён вблизи того места, где ранее находился дом по Немецкой (ныне — Бауманская) улице, в котором, как предполагалось, 26 мая (6 июня) 1799 года родился русский поэт и писатель Александр Сергеевич Пушкин (1799—1837).
  (0.695) На стене соседней школы находится мемориальная доска из красного гранита с бронзовым барельефом Пушкина, созданная скульптором Константином Кошкиным.


## Функция для оценки NDCG@k

In [62]:
import math

def ndcg_k(relevances, rel_ideal, k=10):
    rel_ideal = rel_ideal[:k]    
    discounts = []
    for position in range(1, k + 1):
        discount = math.log2(position + 1)
        discounts.append(discount)
    
    # DCG
    dcg = 0
    for i in range(k):
        gain = relevances[i] / discounts[i]
        dcg += gain
    
    # IDCG
    idcg = 0
    for i in range(k):
        gain = rel_ideal[i] / discounts[i]
        idcg += gain
    
    return dcg / idcg if idcg > 0 else 0

## Результаты рассчетов NDCG@k

### Запрос 1:

In [150]:
print("custom_tf:", ndcg_k(q_custom_tf[0], rel_q_sorted[0]))
print("custom_tf_idf:", ndcg_k(q_custom_tf_idf[0], rel_q_sorted[0]))
print("unigram_tf:", ndcg_k(q_unigram_tf[0], rel_q_sorted[0]))
print("unigram_tf_idf:", ndcg_k(q_unigram_tf_idf[0], rel_q_sorted[0]))
print("bpe_tf:", ndcg_k(q_bpe_tf[0], rel_q_sorted[0]))
print("bpe_tf_idf:", ndcg_k(q_bpe_tf_idf[0], rel_q_sorted[0]))
print("fasttext_tf:", ndcg_k(q_fasttext_tf[0], rel_q_sorted[0]))
print("fasttext_tf_idf:", ndcg_k(q_fasttext_tf_idf[0], rel_q_sorted[0]))

custom_tf: 0.5373331474208121
custom_tf_idf: 0.48950566909605553
unigram_tf: 0.5531793403978953
unigram_tf_idf: 0.46914050927965023
bpe_tf: 0.6583491202824602
bpe_tf_idf: 0.544077240031572
fasttext_tf: 0.5273975579448754
fasttext_tf_idf: 0.4948620750035186


По метрике NDCG@10 лучший результат показала модель bpe_tf: по запросам видно, что уже среди четырех первых выданных предложений есть два предложения с оценкой 2. Скорее всего BPE сильнее других моделей разделил слова на токены и за счет этого вперед выдвинулись релевантные предложения, так как вывелось то, что учитывало все слова из запроса. Остальные модели в целом показали похожие результаты: хоть и релевантные предложения не встретились подряд в самом начале, они имели тенденцию встречаться среди первых 10 предложений в выдаче.

### Запрос 2:

In [151]:
print("custom_tf:", ndcg_k(q_custom_tf[1], rel_q_sorted[1]))
print("custom_tf_idf:", ndcg_k(q_custom_tf_idf[1], rel_q_sorted[1]))
print("unigram_tf:", ndcg_k(q_unigram_tf[1], rel_q_sorted[1]))
print("unigram_tf_idf:", ndcg_k(q_unigram_tf_idf[1], rel_q_sorted[1]))
print("bpe_tf:", ndcg_k(q_bpe_tf[1], rel_q_sorted[1]))
print("bpe_tf_idf:", ndcg_k(q_bpe_tf_idf[1], rel_q_sorted[1]))
print("fasttext_tf:", ndcg_k(q_fasttext_tf[1], rel_q_sorted[1]))
print("fasttext_tf_idf:", ndcg_k(q_fasttext_tf_idf[1], rel_q_sorted[1]))

custom_tf: 0.15423215925316425
custom_tf_idf: 0.29173569656163517
unigram_tf: 0.24445218882151876
unigram_tf_idf: 0.30556523602689845
bpe_tf: 0.2851942202917719
bpe_tf_idf: 0.281245909492015
fasttext_tf: 0.09456701548023277
fasttext_tf_idf: 0.09456701548023277


Здесь лучший результат показала модель unigram_tf_idf c небольшим отрывом от моделей с BPE и custom_tf_idf. В целом здесь модели с tf-idf показывали немного более высокие результаты: это могло быть связано с тем, что уже предложений по тематике этого запроса больше, следовательно сложнее "выцепить" максимально релевантные, что немного облегчает tf_idf, придавая большее значение более редким словам из запроса.

### Запрос 3:

In [152]:
print("custom_tf:", ndcg_k(q_custom_tf[2], rel_q_sorted[2]))
print("custom_tf_idf:", ndcg_k(q_custom_tf_idf[2], rel_q_sorted[2]))
print("unigram_tf:", ndcg_k(q_unigram_tf[2], rel_q_sorted[2]))
print("unigram_tf_idf:", ndcg_k(q_unigram_tf_idf[2], rel_q_sorted[2]))
print("bpe_tf:", ndcg_k(q_bpe_tf[2], rel_q_sorted[2]))
print("bpe_tf_idf:", ndcg_k(q_bpe_tf_idf[2], rel_q_sorted[2]))
print("fasttext_tf:", ndcg_k(q_fasttext_tf[2], rel_q_sorted[2]))
print("fasttext_tf_idf:", ndcg_k(q_fasttext_tf_idf[2], rel_q_sorted[2]))

custom_tf: 0.07930007128354419
custom_tf_idf: 0.12781765803276543
unigram_tf: 0.04353781328125615
unigram_tf_idf: 0.04728350774011639
bpe_tf: 0.0
bpe_tf_idf: 0.040742031470253126
fasttext_tf: 0.1716830951068149
fasttext_tf_idf: 0.17886726863615138


Модели с Fasttext показали лучшие результаты, усредняющий эффект позволил fasttext выделить нужные предложения: в корпусе много подходящих предложений, но только учитывая смысл мы смогли выделить релевантные.

**Выводы:** 

Анализ показал, что результаты поиска сильно зависели от размера релевантного текста: для первого запроса, основанного на небольшой статье, NDCG@10 был в целом выше, так как моделям было проще выбрать из ограниченного числа предложений. При этом эффективность tf-idf варьировалась: он показал себя хуже простого tf на первом запросе, но был увереннее на втором и третьем, где в крупных статьях нужно было отсеять менее релевантные предложения за счёт учёта редкости слов.